### Создание датасета для бенчмарка

In [ ]:
import pandas as pd
from datasets import Dataset

В датасет включен код следующих проектов:

- [serilog](https://github.com/serilog/serilog)
- [Swashbuckle.AspNetCore](https://github.com/domaindrivendev/Swashbuckle.AspNetCore)
- [protobuf](https://github.com/protocolbuffers/protobuf)

Код проектов обработан утилитой BuildDataset, которая удаляет из исходных текстов docstring. На выходе утилита создает файл xml, в котором для каждого docstring указан тип/метод/функция, к которым он относится, и имя файла, где определены соответствующие тип/метод/функция.

In [ ]:
# читаем выход утилиты BuildDataset
ds1 = pd.read_xml("serilog.xml")
ds2 = pd.read_xml("swagger.xml")
ds3 = pd.read_xml("protobuf.xml")

# объединяем все датасеты
ds = pd.concat([ds1, ds2, ds3]).drop_duplicates()

Формируем три датасета для бенчмарка:
- `doc_ds` - файлы исходного кода (документы):
    - `doc_id` - идентификатор документа
    - `file` - имя файла
    - `content` - содержимое файла (с удаленными docstring's)
- `queries_ds` - запросы к документам. В качестве запросов выступают docstring
    - `query_id` - идентификатор запроса
    - `query` - текст запроса
    - `type` - тип элемента кода, к которому относился docstring (тип, либо метод/функция)
- `qrels` - корректные пары запрос/документ
    - `query_id` - идентификатор запроса
    - `doc_id` - идентификатор документа


In [ ]:
doc_ids = {k: 'doc_' + str(v) for v, k in enumerate(ds['file'].unique())}
ds['doc_id'] = ds.file.apply(lambda x: doc_ids[x])

ds['query'] = ds.apply(lambda x: f'[{'Type' if x['type'] == 'T' else 'Type member'}] {x['summary']}', axis=1)
qids = {k: 'query_' + str(v) for v, k in enumerate(ds['query'].unique())}
ds['query_id'] = ds['query'].apply(lambda x: qids[x])

qrels = ds[['query_id', 'doc_id']].drop_duplicates()

def _read_file(path: str):
    with open(path, 'rt', encoding='utf-8') as f:
        return f.read()

doc_ds = ds[['doc_id', 'file']].drop_duplicates()
doc_ds['content'] = doc_ds['file'].apply(_read_file)

queries_ds = ds[['query_id', 'query', 'summary', 'type']].drop_duplicates().rename(columns={'summary': 'docstring'})

Сохраняем датасеты на huggingface:

In [ ]:
hf_qrels = Dataset.from_pandas(qrels, preserve_index=False)
hf_docs = Dataset.from_pandas(doc_ds, preserve_index=False)
hf_queries = Dataset.from_pandas(queries_ds, preserve_index=False)

hf_qrels.push_to_hub('kmvi/code-ir-dataset', config_name='qrels')
hf_docs.push_to_hub('kmvi/code-ir-dataset', config_name='docs')
hf_queries.push_to_hub('kmvi/code-ir-dataset', config_name='queries')

Датасет досупен по [ссылке](https://huggingface.co/datasets/kmvi/code-ir-dataset).